# RamanBench — Benchmarking a New Model

This notebook shows how to evaluate a new model against the 28 precomputed baselines.

## Ecosystem

| Resource | Link |
|---|---|
| **raman-data** (datasets) | [GitHub](https://github.com/ml-lab-htw/raman_data) · [PyPI](https://pypi.org/project/raman-data/) |
| **raman-bench** (this package) | [GitHub](https://github.com/ml-lab-htw/RamanBench) · [PyPI](https://pypi.org/project/raman-bench/) |
| **Live Leaderboard** | [HuggingFace Space](https://huggingface.co/spaces/ml-lab-htw/RamanBench) |
| **Paper** (NeurIPS 2026) | [arXiv TBD](https://arxiv.org/abs/TBD) |

In [ ]:
# !pip install raman-bench raman-data scikit-learn

## Option A: Scikit-learn compatible model (simplest)

In [ ]:
from raman_bench import Leaderboard
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestClassifier

# Load precomputed baselines
lb = Leaderboard.from_precomputed()

print('Current #1 model:', lb.rank().iloc[0]['Model'])

In [ ]:
# PLSRegression fails on tiny datasets when n_components > n_train_samples.
# Wrap it to clip n_components automatically.
from sklearn.base import BaseEstimator, RegressorMixin

class AdaptivePLS(BaseEstimator, RegressorMixin):
    def __init__(self, n_components=20):
        self.n_components = n_components
    def fit(self, X, y):
        n = min(self.n_components, X.shape[0] - 1, X.shape[1])
        self._model = PLSRegression(n_components=max(1, n)).fit(X, y)
        return self
    def predict(self, X):
        return self._model.predict(X)

results = lb.evaluate_and_add(
    model_name="PLS-20",
    model=AdaptivePLS(n_components=20),
    task="regression",
    seeds=3,
)
print(f"Evaluated on {len(results)} (dataset, seed) combinations")


In [ ]:
# View updated regression ranking
ranked = lb.rank(task="regression")
pls_row = ranked[ranked["Model"] == "PLS-20"]
pls_rank = pls_row.index[0] + 1
print(f"PLS-20 ranked #{pls_rank} out of {len(ranked)} models (regression)")
ranked.head(10)


In [ ]:
import matplotlib.pyplot as plt

fig = lb.plot(task='overall', n_top=35)
plt.show()

## Option B: Adding results from a pre-existing run

If you have already run your model and computed metrics, you can add the results directly:

In [ ]:
import pandas as pd
from raman_bench import Leaderboard

# Your pre-computed metrics DataFrame
# Must have columns: seed, key, and metric columns (rmse/r2 for regression, f1_score for classification)
my_metrics = pd.DataFrame({
    'seed': [0, 1, 2],
    'key': ['amino_acids_glycine_0', 'amino_acids_glycine_0', 'amino_acids_glycine_0'],
    'rmse': [0.05, 0.048, 0.052],
    'r2': [0.95, 0.96, 0.94],
})

lb = Leaderboard.from_precomputed()
lb.add_results('My Existing Model', my_metrics)
print(lb.rank().tail(30))

## Option C: Full benchmark run via AutoGluon

For AutoGluon-integrated models, use the CLI:

```bash
# Run a specific model
raman-bench run --config configs/benchmark_v0.1.json --model MYMODEL

# Then load results and compare
```

See [CONTRIBUTING.md](../CONTRIBUTING.md) for instructions on implementing a fully
integrated AutoGluon model.